# Análisis Exploratorio para Series de Tiempo
### Tendencia, estacionalidad y ciclos, con Plotly

`Fase 1 · Video 2` — serie **Forecasting de Ventas de la A a la Z**

En el **Video 1** generamos el dataset sintético (4 años, 50 SKUs, 4 canales, con tendencia,
estacionalidad, eventos y promociones). Antes de modelar hay que **mirarlo**: aquí exploramos sus
patrones y salimos con hipótesis concretas que los siguientes videos convertirán en evidencia.

> Usamos **Plotly** (zoom, hover, HTML para compartir) y **plotly-resampler** para que 4 años de
> datos diarios no congelen el navegador.

---
## 1. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly_resampler import FigureResampler
from pathlib import Path

BLUE, RED, GREEN, PURPLE, ORANGE = "#2563EB", "#DC2626", "#16A34A", "#7C3AED", "#EA580C"
PALETTE_5 = [BLUE, GREEN, ORANGE, PURPLE, RED]

HTML_DIR = Path("../output/eda_html")
HTML_DIR.mkdir(parents=True, exist_ok=True)
print("Librerías cargadas")

In [ ]:
def find_csv(filename="sales_history.csv", max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / "output" / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")


csv_path = find_csv()
df_raw = pd.read_csv(csv_path, nrows=2)
date_col = next(c for c in df_raw.columns if "date" in c.strip().lower())
df = pd.read_csv(csv_path, parse_dates=[date_col])
df.rename(columns={date_col: "date"}, inplace=True)
df.sort_values("date", inplace=True)

print(f"✅ {len(df):,} filas | {df['date'].min().date()} → {df['date'].max().date()}")
print(
    f"   {df['sku_id'].nunique()} SKUs | {df['channel'].nunique()} canales | {df['category'].nunique()} categorías"
)

In [ ]:
def save_html(fig, name):
    out = HTML_DIR / f"{name}.html"
    fig.write_html(out, include_plotlyjs="inline", full_html=True)
    print(f"Guardado: {out.name}")

---
## 2. Serie agregada
La primera mirada: revenue diario con media móvil, range slider y botones de zoom temporal.

In [ ]:
daily = df.groupby("date", as_index=False).agg(
    revenue=("revenue", "sum"), units=("units_sold", "sum")
)
rolling = daily["revenue"].rolling(30, center=True).mean()

fig = FigureResampler(go.Figure())
fig.add_trace(
    go.Scattergl(name="Revenue diario", line=dict(color=BLUE, width=1.2)),
    hf_x=daily["date"],
    hf_y=daily["revenue"],
)
fig.add_trace(
    go.Scattergl(name="Media móvil 30 días", line=dict(color=RED, width=2.5)),
    hf_x=daily["date"],
    hf_y=rolling,
)
fig.update_layout(
    title="Revenue Diario Total — Serie Completa",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=[
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(step="all", label="Todo"),
            ]
        ),
    ),
    yaxis=dict(title="Revenue ($)", tickformat="$,.0f"),
)
fig.show()
save_html(fig, "01_serie_total")

---
## 3. Calendar heatmap
Cada celda es un día (semana del año × día de la semana). Los picos saltan a la vista donde una
línea temporal los escondería.

In [ ]:
# Pivot: semana del año × día de la semana, un heatmap por año
years_to_plot = sorted(daily["date"].dt.year.unique())
n_years = len(years_to_plot)

fig = make_subplots(
    rows=n_years,
    cols=1,
    subplot_titles=[f"Año {y}" for y in years_to_plot],
    vertical_spacing=0.05,
)

weekday_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

for i, year in enumerate(years_to_plot, start=1):
    d_year = daily[daily["date"].dt.year == year].copy()
    d_year["week"] = d_year["date"].dt.isocalendar().week
    d_year["weekday"] = d_year["date"].dt.weekday

    pivot = d_year.pivot_table(
        index="weekday", columns="week", values="revenue", aggfunc="sum"
    )

    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=weekday_labels,
            colorscale="RdBu_r",
            zmin=daily["revenue"].quantile(0.05),
            zmax=daily["revenue"].quantile(0.99),
            hovertemplate="Semana %{x}<br>%{y}<br>Revenue: $%{z:,.0f}<extra></extra>",
            showscale=(i == 1),
            colorbar=dict(title="Revenue", tickformat="$,.0f"),
        ),
        row=i,
        col=1,
    )
    fig.update_xaxes(title_text="Semana del año" if i == n_years else "", row=i, col=1)

fig.update_layout(
    title="Calendar Heatmap — Revenue Diario por Año",
    template="plotly_white",
    height=250 * n_years,
)

fig.show()
save_html(fig, "02_calendar_heatmap")

---
## 4. Comparación año a año
Superponer los años en el mismo eje revela el patrón: si las curvas coinciden hay estacionalidad;
si crecen, tendencia. Spoiler: esta serie tiene ambas.

In [ ]:
daily["day_of_year"] = daily["date"].dt.dayofyear
daily["year"] = daily["date"].dt.year

fig = go.Figure()

for year, color in zip(years_to_plot, PALETTE_5):
    d_year = daily[daily["year"] == year]
    fig.add_trace(
        go.Scatter(
            x=d_year["day_of_year"],
            y=d_year["revenue"],
            name=str(year),
            line=dict(color=color, width=1.5),
            opacity=0.85,
            hovertemplate=f"{year} - Día %{{x}}<br>Revenue: $%{{y:,.0f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Comparación Año a Año — Revenue Diario Superpuesto",
    template="plotly_white",
    height=500,
    hovermode="x unified",
    xaxis=dict(title="Día del año"),
    yaxis=dict(title="Revenue ($)", tickformat="$,.0f"),
    legend=dict(title="Año"),
)

fig.show()
save_html(fig, "03_year_over_year")

---
## 5. Evolución por canal
Área apilada (revenue absoluto) y 100% apilada (participación). Una responde *cuánto vende cada
canal*; la otra, *quién gana o pierde terreno*.

In [ ]:
# Agregación mensual por canal (mensual para que no sea ruido)
monthly_ch = df.groupby([pd.Grouper(key="date", freq="ME"), "channel"], as_index=False)[
    "revenue"
].sum()
monthly_pivot = monthly_ch.pivot(
    index="date", columns="channel", values="revenue"
).fillna(0)
channels = monthly_pivot.columns.tolist()
color_map = {ch: PALETTE_5[i % len(PALETTE_5)] for i, ch in enumerate(channels)}

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        "Revenue Mensual por Canal (absoluto)",
        "Participación Relativa (% del total)",
    ),
    vertical_spacing=0.12,
)

for ch in channels:
    fig.add_trace(
        go.Scatter(
            x=monthly_pivot.index,
            y=monthly_pivot[ch],
            name=ch,
            mode="lines",
            stackgroup="abs",
            line=dict(width=0.5, color=color_map[ch]),
            hovertemplate=f"{ch}<br>%{{x|%b %Y}}<br>$%{{y:,.0f}}<extra></extra>",
        ),
        row=1,
        col=1,
    )

pct_pivot = monthly_pivot.div(monthly_pivot.sum(axis=1), axis=0) * 100
for ch in channels:
    fig.add_trace(
        go.Scatter(
            x=pct_pivot.index,
            y=pct_pivot[ch],
            name=ch,
            mode="lines",
            stackgroup="pct",
            line=dict(width=0.5, color=color_map[ch]),
            showlegend=False,
            hovertemplate=f"{ch}<br>%{{x|%b %Y}}<br>%{{y:.1f}}%<extra></extra>",
        ),
        row=2,
        col=1,
    )

fig.update_yaxes(title_text="Revenue ($)", tickformat="$,.0f", row=1, col=1)
fig.update_yaxes(title_text="% del total", ticksuffix="%", row=2, col=1)
fig.update_layout(
    template="plotly_white",
    height=750,
    hovermode="x unified",
    title="Evolución por Canal de Venta",
)

fig.show()
save_html(fig, "04_channel_evolution")

---
## 6. Categorías y SKUs
El treemap responde de un vistazo *¿de dónde viene el revenue?*, y el top de SKUs anticipa la
segmentación por tiers del Video 7.

In [ ]:
sku_summary = df.groupby(["category", "sku_id"], as_index=False).agg(
    revenue=("revenue", "sum"), units=("units_sold", "sum")
)

fig = px.treemap(
    sku_summary,
    path=[px.Constant("Total"), "category", "sku_id"],
    values="revenue",
    color="revenue",
    color_continuous_scale="Blues",
    hover_data={"units": ":,", "revenue": ":$,.0f"},
)
fig.update_traces(
    hovertemplate="<b>%{label}</b><br>Revenue: $%{value:,.0f}<extra></extra>",
    textinfo="label+percent parent",
)
fig.update_layout(
    title="Treemap — Revenue por Categoría → SKU (4 años acumulados)",
    template="plotly_white",
    height=600,
)

fig.show()
save_html(fig, "05_treemap_categoria_sku")

In [ ]:
top15 = sku_summary.nlargest(15, "revenue")

fig = px.bar(
    top15.sort_values("revenue"),
    x="revenue",
    y="sku_id",
    color="category",
    orientation="h",
    color_discrete_sequence=PALETTE_5,
    text="revenue",
)
fig.update_traces(
    texttemplate="$%{x:,.0f}",
    textposition="outside",
)
fig.update_layout(
    title="Top 15 SKUs por Revenue Total (4 años)",
    template="plotly_white",
    height=600,
    xaxis=dict(title="Revenue ($)", tickformat="$,.0f"),
    yaxis=dict(title=""),
)

fig.show()
save_html(fig, "06_top_skus")

---
## 7. Eventos y picos
Marcamos los eventos comerciales del dataset sobre la serie. Esa relación evento → pico es el
anticipo del Video 6 (outliers).

In [ ]:
events_df = (
    df[df["event"] != "Regular"]
    .groupby("event", as_index=False)
    .agg(start=("date", "min"), end=("date", "max"))
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=daily["date"],
        y=daily["revenue"],
        name="Revenue diario",
        line=dict(color=BLUE, width=1),
        hovertemplate="%{x|%Y-%m-%d}<br>$%{y:,.0f}<extra></extra>",
    )
)

event_colors = {
    "Buen Fin": "#DC2626",
    "Black Fri": "#7C3AED",
    "Hot Sale": "#EA580C",
    "Cyber": "#16A34A",
    "Navidad": "#0EA5E9",
    "Reyes": "#F59E0B",
    "Día Madres": "#EC4899",
    "San Valentín": "#F43F5E",
}

for _, ev in events_df.iterrows():
    color = event_colors.get(ev["event"], "#94A3B8")
    # Cada ocurrencia del evento (un rectángulo por instancia anual)
    occurrences = (
        df[df["event"] == ev["event"]]
        .groupby(df[df["event"] == ev["event"]]["date"].dt.year)["date"]
        .agg(["min", "max"])
    )
    for _, occ in occurrences.iterrows():
        fig.add_vrect(
            x0=occ["min"],
            x1=occ["max"],
            fillcolor=color,
            opacity=0.15,
            layer="below",
            line_width=0,
            annotation_text=ev["event"] if occ.name == occurrences.index[0] else "",
            annotation_position="top left",
            annotation_font_size=9,
        )

fig.update_layout(
    title="Revenue Diario con Eventos Comerciales Marcados",
    template="plotly_white",
    height=550,
    hovermode="x unified",
    xaxis=dict(
        title="",
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=[
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(step="all", label="Todo"),
            ]
        ),
    ),
    yaxis=dict(title="Revenue ($)", tickformat="$,.0f"),
)

fig.show()
save_html(fig, "07_eventos_comerciales")

regular_mean = (
    df[df["event"] == "Regular"]["revenue"].sum()
    / df[df["event"] == "Regular"]["date"].nunique()
)
lift = (
    df[df["event"] != "Regular"]
    .groupby("event")
    .apply(lambda g: g["revenue"].sum() / g["date"].nunique(), include_groups=False)
    .sort_values(ascending=False)
    .rename("revenue_diario_promedio")
)
lift_pct = ((lift / regular_mean) - 1) * 100

print(f"Revenue diario promedio en días Regulares: ${regular_mean:,.0f}\n")
print("Lift por evento (vs día Regular):")
for ev, pct in lift_pct.items():
    print(f"  {ev:<14}: ${lift[ev]:>12,.0f}/día  →  {pct:+.1f}%")

---
## 8. Exportar para compartir
Cada gráfico queda como HTML autocontenido en `output/eda_html/`: se abre en cualquier navegador,
sin Python.

In [ ]:
html_files = sorted(HTML_DIR.glob("*.html"))
print(f"Visualizaciones exportadas ({len(html_files)} archivos):")
for f in html_files:
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:<35} ({size_kb:>7.1f} KB)")
print(f"\nUbicación: {HTML_DIR.resolve()}")

---
## 9. Lo que vimos → lo que probaremos
El EDA genera hipótesis; los siguientes videos las miden.

| Hallazgo visual | Se formaliza en |
|---|---|
| Tendencia creciente año a año | **Video 3** — ADF/KPSS |
| Picos estacionales de fin de año | **Video 4** — descomposición STL |
| Estructura que se repite cada año | **Video 5** — ACF en lag 52 |
| Días muy por encima de la media | **Video 6** — outliers |
| Cambio de mezcla entre canales | **Video 7** — tiers de forecasting |

---
### Próximo video
**Video 3 — Estacionariedad: ADF, KPSS y transformaciones.** Vimos que hay tendencia; ahora la
probamos formalmente y vemos qué hacer cuando los tests se contradicen.